In [2]:
import qi
import os
import sys
import time
import threading

from naoqi import ALProxy

# Bot's IP address and port
ip = '10.104.23.185'
port = 9559

## Transfer and Receiving File Functions

In [3]:
!pip install paramiko
import paramiko
def transfer_file(remote_path="/home/nao/microphones/recording.wav", local_path="recordings/recording.wav"):
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(ip, username='nao', password='nao')

    sftp = ssh.open_sftp()
    sftp.get(remote_path, local_path)

    sftp.close()
    ssh.close()
    print("File transfered")

def receive_file(local_path="recordings/recording.wav", remote_path="/home/nao/microphones/recording.wav"):
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(ip, username='nao', password='nao')

    sftp = ssh.open_sftp()
    sftp.put(local_path, remote_path)

    sftp.close()
    ssh.close()

DEPRECATION: Python 2.7 reached the end of its life on January 1st, 2020. Please upgrade your Python as Python 2.7 is no longer maintained. A future version of pip will drop support for Python 2.7. More details about Python 2 support in pip, can be found at https://pip.pypa.io/en/latest/development/release-process/#python-2-support
     |████████████████████████████████| 213 kB 5.1 MB/s eta 0:00:01
     |████████████████████████████████| 2.6 MB 21.4 MB/s eta 0:00:01
     |████████████████████████████████| 964 kB 28.4 MB/s eta 0:00:01
     |████████████████████████████████| 59 kB 643 kB/s  eta 0:00:01
     |████████████████████████████████| 390 kB 16.2 MB/s eta 0:00:01
     |████████████████████████████████| 118 kB 14.5 MB/s eta 0:00:01
You should consider upgrading via the '/usr/local/bin/python -m pip install --upgrade pip' command.


/usr/local/lib/python2.7/site-packages/paramiko/transport.py:33: CryptographyDeprecationWarning: Python 2 is no longer supported by the Python core team. Support for it is now deprecated in cryptography, and will be removed in the next release.
  from cryptography.hazmat.backends import default_backend


In [6]:
receive_file(local_path="qr_codes/image.png", remote_path="/home/nao/.local/share/PackageManager/apps/.lastUploadedChoregrapheBehavior/html/image.png")

In [13]:
launchAndStopBehavior(service, ".lastUploadedChoregrapheBehavior/behavior_1")

## Audio Recording

In [4]:
def eye_color(status="ListenOn", rgb = None):
    leds = ALProxy("ALLeds", ip, port)

    if rgb is None:
        if status == "ListenOn":
            leds.fadeRGB("AllLeds", "yellow", 1)
        elif status == "SpeechDetected":
            leds.fadeRGB("AllLeds", "green", 8)
        elif status == "EndOfProcess":
            leds.fadeRGB("AllLeds", "white", 1)

    else:
        leds.fadeRGB("FaceLEDS", rgb, 1)


# Callback function for timer
def timer_callback(timer_cb):
    print("Timer reached")
    timer_cb.set()

# Records Audio when speech is detected
def record_audio(timer=None, path_name="/home/nao/microphones/recording.wav", debug=False):
    path_name = os.path.join(path_name)
    # Create Connection Proxies
    recorder = ALProxy("ALAudioRecorder", ip, port)
    speech_recogniser = ALProxy("ALSpeechRecognition", ip, port)
    sound_detector = ALProxy("ALSoundDetection", ip, port)
    memory = ALProxy("ALMemory", ip, port)

    speech_recogniser.subscribe("speech_recognition")
    speech_recogniser.setLanguage("English")
    vocabulary = ["pepper", "hello"]
    speech_recogniser.setVocabulary(vocabulary, False)

    # Stop recording when speech stops being detected or timer is reached
    timer_cb = threading.Event()
    if timer is not None:
        threading.Timer(timer, timer_callback, timer_cb).start()

    # Start recording
    while True:
        time.sleep(0.5)
        status = memory.getData("ALSpeechRecognition/Status")

        # Print status
        if debug:
            print(status)

        # Start recording when speech is detected
        if status == "SpeechDetected":
            eye_color(status)
            print("Recording is starting...")
            recorder.startMicrophonesRecording(path_name, "wav", 16000, (0,0,1,0))
            break

        # Stop recording when timer is reached
        if timer_cb.is_set():
            break

    if timer_cb.is_set():
        return

    while True:
        time.sleep(0.5)
        status = memory.getData("ALSpeechRecognition/Status")

        # Stop recording when speech stops being detected
        if status == "EndOfProcess":
            print("Recording is stopping...")
            recorder.stopMicrophonesRecording()
            speech_recogniser.unsubscribe("speech_recognition")
            eye_color(status)
            break

        # Stop recording when timer is reached
        if timer_cb.is_set():
            recorder.stopMicrophonesRecording()
            speech_recogniser.unsubscribe("speech_recognition")
            leds = ALProxy("ALLeds", ip, port)
            leds.fadeRGB("AllLeds", "White", 1)
            break


def record_audio_sd(timer=None, path_name="/home/nao/microphones/recording.wav", debug=False):
    path_name = os.path.join(path_name)
    # Create Connection Proxies
    recorder = ALProxy("ALAudioRecorder", ip, port)
    sound_detector = ALProxy("ALSoundDetection", ip, port)
    memory = ALProxy("ALMemory", ip, port)
    
    timer_cb = threading.Event()

    sound_detector.subscribe("sound_detector")
    sound_detector.setParameter("Sensitivity", 0.8)

    # Record audio when sound is detected
    if timer is not None:
        threading.Timer(timer, timer_callback, [timer_cb]).start()

    # Start recording
    while True:
        time.sleep(0.5)
        status = memory.getData("SoundDetected")

        # Print status
        if debug:
            print(status)

        # Start recording when sound is detected
        if status is not None:
            print("Recording is starting...")
            recorder.startMicrophonesRecording(path_name, "wav", 16000, (0,0,1,0))
            eye_color("SpeechDetected")
            break

        # Stop recording when timer is reached
        if timer_cb.is_set():
            break

    if timer_cb.is_set():
        return
    
    sound_detector.setParameter("Sensitivity", 0.7)
    while True:
        time.sleep(3)
        status = memory.getData("SoundDetected")

        if debug:
            print("Status: ", status)

        # Stop recording when sound stops being detected
        if status is None:
            print("Recording is stopping...")
            recorder.stopMicrophonesRecording()
            sound_detector.unsubscribe("sound_detector")
            leds = ALProxy("ALLeds", ip, port)
            leds.fadeRGB("AllLeds", "white", 1)
            break

        # Stop recording when timer is reached
        if timer_cb:
            print("Recording is stopping...")
            recorder.stopMicrophonesRecording()
            sound_detector.unsubscribe("sound_detector")
            leds = ALProxy("ALLeds", ip, port)
            leds.fadeRGB("AllLeds", "white", 1)
            break

## Speech to Text

In [ ]:
!pip install speechrecognition==3.8.1
import speech_recognition as sr
def convert_wav_to_text(audio_path):
    r = sr.Recognizer()
    with sr.AudioFile(audio_path) as source:
        audio_data = r.record(source)
    try:
        text = r.recognize_google(audio_data)
        return text
    except sr.UnknownValueError:
        return "Speech recognition could not understand audio"
    except sr.RequestError as e:
        return "Could not request results from Google Speech Recognition service: " + str(e)
    except Exception as e:
        return "An error occurred during speech recognition: " + str(e)

DEPRECATION: Python 2.7 reached the end of its life on January 1st, 2020. Please upgrade your Python as Python 2.7 is no longer maintained. A future version of pip will drop support for Python 2.7. More details about Python 2 support in pip, can be found at https://pip.pypa.io/en/latest/development/release-process/#python-2-support
     |████████████████████████████████| 32.8 MB 6.7 MB/s eta 0:00:011
You should consider upgrading via the '/usr/local/bin/python -m pip install --upgrade pip' command.


## Testing

In [48]:
recorder = ALProxy("ALAudioRecorder", ip, port)
recorder.stopMicrophonesRecording()

In [65]:
record_audio_sd(timer=15, debug=False)
transfer_file()
result = convert_wav_to_text("recordings/recording.wav")
result

[[6165, 0, 1.0, 0]]
Recording is starting...
('Status: ', [[261, 0, 1.0, 0]])
Recording is stopping...
File transfered
Timer reached


**Method 2**

In [ ]:
!pip install pyftplib
import pyftplib
from pyftplib import FTPServer
from pyftplib.handlers import FTPHandler
from pyftplib.authorizers import DummyAuthorizer
import threading

def ftp_transfer_file():
    remote_path = "/home/nao/microphones/recording.wav"
    authorizer = DummyAuthorizer()
    authorizer.add_user("nao", "nao", remote_path, perm="elradfmw")
    handler = FTPHandler
    handler.authorizer = authorizer
    server = FTPServer((ip, 21), handler)
    server_thread = threading.Thread(target=server.serve_forever)
    server_thread.daemon = True
    server_thread.start()

# Tablet

In [5]:
tablet_service = ALProxy("ALTabletService", ip, port)
tablet_service.wakeUp()
tablet_service.resetTablet()
tablet_service.enableWifi()

In [57]:
tablet_service.getWifiStatus()

'DISCONNECTED'

In [58]:
# tablet_service.configureWifi("ACpg2DpVU3rCo)z5", "Deakin IoT", "WPA2-Personal")
tablet_service.connectWifi("Deakin IoT")

True

In [52]:
tablet_service.getWifiStatus()

'DISCONNECTED'

In [32]:
# Pepper recieves html page from laptop
receive_file(local_path="/path/to/html/on/laptop/page.html", remote_path="home/nao/.local/share/PackageManager/apps/robot-page/html/page.html")

# Pepper loads html page
tablet_service.loadUrl("http://198.18.0.1/page.html")
tablet_service.showWebview()

## Temperature Management

In [11]:
!pip install pandas

from naoqi import qi
from pandas import DataFrame
from IPython.display import clear_output, display, Javascript
import time

idling = [
    'animations/Stand/Waiting/BackRubs_1', 
    'animations/Stand/Waiting/Bandmaster_1', 
    'animations/Stand/Waiting/Binoculars_1', 
    'animations/Stand/Waiting/Drink_1', 
    'animations/Stand/Waiting/Fitness_1', 'animations/Stand/Waiting/Fitness_2', 'animations/Stand/Waiting/Fitness_3', 
    'animations/Stand/Waiting/FunnyDancer_1', 
    'animations/Stand/Waiting/HideEyes_1', 
    'animations/Stand/Waiting/HideHands_1', 
    'animations/Stand/Waiting/Innocent_1', 
    'animations/Stand/Waiting/Knight_1',  
    'animations/Stand/Waiting/KungFu_1', 
    'animations/Stand/Waiting/LookHand_1', 'animations/Stand/Waiting/LookHand_2', 
    'animations/Stand/Waiting/LoveYou_1',  
    'animations/Stand/Waiting/PlayHands_1', 'animations/Stand/Waiting/PlayHands_2', 'animations/Stand/Waiting/PlayHands_3', 
    'animations/Stand/Waiting/Relaxation_1', 'animations/Stand/Waiting/Relaxation_2', 'animations/Stand/Waiting/Relaxation_3', 'animations/Stand/Waiting/Relaxation_4', 
    'animations/Stand/Waiting/Rest_1', 
    'animations/Stand/Waiting/ScratchBack_1', 
    'animations/Stand/Waiting/ScratchBottom_1', 
    'animations/Stand/Waiting/ScratchEye_1', 
    'animations/Stand/Waiting/ScratchHand_1', 
    'animations/Stand/Waiting/ScratchHead_1', 
    'animations/Stand/Waiting/ScratchLeg_1', 
    'animations/Stand/Waiting/ScratchTorso_1', 
    'animations/Stand/Waiting/ShowMuscles_1', 'animations/Stand/Waiting/ShowMuscles_2', 'animations/Stand/Waiting/ShowMuscles_3', 'animations/Stand/Waiting/ShowMuscles_4', 'animations/Stand/Waiting/ShowMuscles_5', 
    'animations/Stand/Waiting/ShowSky_1', 'animations/Stand/Waiting/ShowSky_2', 
    'animations/Stand/Waiting/Stretch_1', 'animations/Stand/Waiting/Stretch_2',
    'animations/Stand/Waiting/Think_1', 'animations/Stand/Waiting/Think_2', 'animations/Stand/Waiting/Think_3', 'animations/Stand/Waiting/Think_4', 
    'animations/Stand/Waiting/Waddle_1', 'animations/Stand/Waiting/Waddle_2', 
    'animations/Stand/Waiting/Zombie_1'
]

def launchAndStopBehavior(behavior_mng_service, behavior_name):
    """
    Launch and stop a behavior, if possible.
    """
    # Check that the behavior exists.
    if (behavior_mng_service.isBehaviorInstalled(behavior_name)):
        # Check that it is not already running.
        if (not behavior_mng_service.isBehaviorRunning(behavior_name)):
            # Launch behavior. This is a blocking call, use _async=True if you do not
            # want to wait for the behavior to finish.
            behavior_mng_service.runBehavior(behavior_name, _async=True)
            time.sleep(0.5)
        else:
            print "Behavior is already running."

    else:
        print "Behavior not found."
    return

    names = behavior_mng_service.getRunningBehaviors()
    print "Running behaviors:"
    print names

    # Stop the behavior.
    if (behavior_mng_service.isBehaviorRunning(behavior_name)):
        behavior_mng_service.stopBehavior(behavior_name)
        time.sleep(1.0)
    else:
        print "Behavior is already stopped."

    names = behavior_mng_service.getRunningBehaviors()
    print "Running behaviors:"
    print names

def print_data(_data):
    clear_output(wait=True)
    print(
        data.apply(
            lambda x: '\033[91m{:.2f}\033[0m'.format(x) if x > 60 else '\033[92m{:.2f}\033[0m'.format(x)
        ).to_string()
    )
    return None

memory_service = session.service("ALMemory")
temprature_data = {
    'datetime': [],
    'HeadYaw': [],
    'HeadPitch': [],
    'LElbowYaw': [],
    'LElbowRoll': [],
    'RElbowYaw': [],
    'RElbowRoll': [],
    'LHand': [],
    'LWristYaw': [],
    'RHand': [],
    'RWristYaw': [],
    'LShoulderPitch': [],
    'LShoulderRoll': [],
    'RShoulderPitch': [],
    'RShoulderRoll': [],
    'HipRoll': [],
    'HipPitch': [],
    'KneePitch': [],
    'WheelFL': [],
    'WheelFR': [],
    'WheelB': [],
}

DEPRECATION: Python 2.7 reached the end of its life on January 1st, 2020. Please upgrade your Python as Python 2.7 is no longer maintained. A future version of pip will drop support for Python 2.7. More details about Python 2 support in pip, can be found at https://pip.pypa.io/en/latest/development/release-process/#python-2-support
You should consider upgrading via the '/usr/local/bin/python -m pip install --upgrade pip' command.


In [ ]:
alert = 0
while True:
    now = time.localtime()
    now = '%d_%d_%d_%d_%d_%d' % (now.tm_year, now.tm_mon, now.tm_mday, now.tm_hour, now.tm_min, now.tm_sec)
    temprature_data['datetime'].append(now)
    
    temprature_data['HeadYaw'].append(memory_service.getData("Device/SubDeviceList/HeadYaw/Temperature/Sensor/Value"))
    temprature_data['HeadPitch'].append(memory_service.getData("Device/SubDeviceList/HeadPitch/Temperature/Sensor/Value"))
    temprature_data['LElbowYaw'].append(memory_service.getData("Device/SubDeviceList/LElbowYaw/Temperature/Sensor/Value"))
    temprature_data['LElbowRoll'].append(memory_service.getData("Device/SubDeviceList/LElbowRoll/Temperature/Sensor/Value"))
    temprature_data['RElbowYaw'].append(memory_service.getData("Device/SubDeviceList/RElbowYaw/Temperature/Sensor/Value"))
    temprature_data['RElbowRoll'].append(memory_service.getData("Device/SubDeviceList/RElbowRoll/Temperature/Sensor/Value"))
    temprature_data['LHand'].append(memory_service.getData("Device/SubDeviceList/LHand/Temperature/Sensor/Value"))
    temprature_data['LWristYaw'].append(memory_service.getData("Device/SubDeviceList/LWristYaw/Temperature/Sensor/Value"))
    temprature_data['RHand'].append(memory_service.getData("Device/SubDeviceList/RHand/Temperature/Sensor/Value"))
    temprature_data['RWristYaw'].append(memory_service.getData("Device/SubDeviceList/RWristYaw/Temperature/Sensor/Value"))
    temprature_data['LShoulderPitch'].append(memory_service.getData("Device/SubDeviceList/LShoulderPitch/Temperature/Sensor/Value"))
    temprature_data['LShoulderRoll'].append(memory_service.getData("Device/SubDeviceList/LShoulderRoll/Temperature/Sensor/Value"))
    temprature_data['RShoulderPitch'].append(memory_service.getData("Device/SubDeviceList/RShoulderPitch/Temperature/Sensor/Value"))
    temprature_data['RShoulderRoll'].append(memory_service.getData("Device/SubDeviceList/RShoulderRoll/Temperature/Sensor/Value"))
    temprature_data['HipRoll'].append(memory_service.getData("Device/SubDeviceList/HipRoll/Temperature/Sensor/Value"))
    temprature_data['HipPitch'].append(memory_service.getData("Device/SubDeviceList/HipPitch/Temperature/Sensor/Value"))
    temprature_data['KneePitch'].append(memory_service.getData("Device/SubDeviceList/KneePitch/Temperature/Sensor/Value"))
    temprature_data['WheelFL'].append(memory_service.getData("Device/SubDeviceList/WheelFL/Temperature/Sensor/Value"))
    temprature_data['WheelFR'].append(memory_service.getData("Device/SubDeviceList/WheelFR/Temperature/Sensor/Value"))
    temprature_data['WheelB'].append(memory_service.getData("Device/SubDeviceList/WheelB/Temperature/Sensor/Value"))
    
    data = DataFrame(temprature_data).iloc[-1, :-1]
    print_data(_data=data)
    raise_alert = (data.max() > 50) & (time.time() - alert > 30)
    alert = time.time() if raise_alert else alert
    display(Javascript('alert("Alert: Value > 50 detected!")')) if raise_alert else None
    time.sleep(1)

In [10]:
from random import choice
import time
from naoqi import qi

session = qi.Session()
session.connect("tcp://" + ip + ":" + str(port))
service = session.service("ALBehaviorManager")

# while True:
#     choosen = choice(idling)
#     launchAndStopBehavior(service, choosen)
#     print('Executing', choosen)
#     time.sleep(20)

## Misc

In [ ]:
file_manager = ALProxy("ALFileManager", ip, port)
shared_folder_name = file_manager.getUserSharedFolderPath()
shared_folder_name

'/home/nao/'

In [ ]:
# Record Video
def record_video(timer=10):
    video_recorder = ALProxy("ALVideoRecorder", ip, port)
    video_recorder.startRecording("/home/nao/recordings/cameras", "test_video")
    time.sleep(timer)
    videoInfo = video_recorder.stopRecording()

    print('Video was saved on the robot: ', videoInfo[1])
    print('Total number of frames: ', videoInfo[0])

('Video was saved on the robot: ', '/home/nao/recordings/cameras/test_video.avi')
('Total number of frames: ', 73)
